<a href="https://colab.research.google.com/github/AbhishekChaganti/MY_REPO/blob/main/video_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch torchvision pillow opencv-python

import cv2
import torch
import numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification
from google.colab import files

# ================= UPLOAD VIDEO =================
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Uploaded: {video_path}")

# ================= LOAD MODEL =================
model_name = "dima806/deepfake_vs_real_image_detection"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Force label mapping: 0 = FAKE, 1 = REAL
FAKE_IDX = 0
REAL_IDX = 1
print(f"Model original labels: {model.config.id2label}")
print(f"Using fixed mapping → 0: FAKE, 1: REAL")
print(f"Device: {device}")

# ================= EXTRACT FRAMES =================
def extract_frames(video_path, frame_rate=4):
    cap = cv2.VideoCapture(video_path)
    frames = []

    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    interval = max(1, int(fps // frame_rate))

    print(f"\nVideo FPS: {fps:.1f} | Total frames: {total_frames} | Sampling every {interval} frames")

    count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(rgb)
        count += 1

    cap.release()
    print(f"Extracted {len(frames)} frames for prediction\n")
    return frames

# ================= PREDICT SINGLE FRAME =================
def predict_frame(frame_np):
    image = Image.fromarray(frame_np)
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    fake_score = float(probs[FAKE_IDX])
    real_score = float(probs[REAL_IDX])
    label = "REAL" if real_score > fake_score else "FAKE"

    return {
        "label": label,
        "fake_score": fake_score,
        "real_score": real_score
    }

# ================= PREDICT ALL FRAMES =================
def predict_video(video_path):
    frames = extract_frames(video_path)

    fake_count = 0
    real_count = 0
    all_fake_scores = []

    print(f"{'Frame':<8} {'Label':<8} {'Fake Score':<14} {'Real Score':<14}")
    print("-" * 46)

    for i, frame in enumerate(frames):
        result = predict_frame(frame)

        if result["label"] == "FAKE":
            fake_count += 1
        else:
            real_count += 1

        all_fake_scores.append(result["fake_score"])

        print(f"{i:<8} {result['label']:<8} {result['fake_score']:<14.4f} {result['real_score']:<14.4f}")

    # ================= SUMMARY =================
    total = len(frames)
    avg_fake = sum(all_fake_scores) / total
    median_fake = sorted(all_fake_scores)[total // 2]

    # Final verdict: majority vote
    final = "FAKE" if fake_count > real_count else "REAL"
    fake_pct = (fake_count / total) * 100
    real_pct = (real_count / total) * 100

    print("\n" + "=" * 46)
    print("           FINAL SUMMARY")
    print("=" * 46)
    print(f"Total frames predicted : {total}")
    print(f"FAKE frames            : {fake_count} ({fake_pct:.1f}%)")
    print(f"REAL frames            : {real_count} ({real_pct:.1f}%)")
    print(f"Avg fake score         : {avg_fake:.4f}")
    print(f"Median fake score      : {median_fake:.4f}")
    print("-" * 46)
    print(f"FINAL VERDICT          : {final}")
    print("=" * 46)

    return {
        "final_verdict": final,
        "total_frames": total,
        "fake_frames": fake_count,
        "real_frames": real_count,
        "fake_pct": round(fake_pct, 2),
        "real_pct": round(real_pct, 2),
        "avg_fake_score": round(avg_fake, 4),
        "median_fake_score": round(median_fake, 4)
    }

# ================= RUN =================
result = predict_video(video_path)